In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Wits_Honours_Project

/content/drive/MyDrive/Wits_Honours_Project


In [ ]:
!pip -q install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 12.1 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/Wits_Honours_Project
!git pull

/content/drive/MyDrive/Wits_Honours_Project
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 810 bytes | 11.00 KiB/s, done.
From https://github.com/Reese1718/Wits_Honours_Project
   165cf64..67b87e6  main       -> origin/main
Updating 165cf64..67b87e6
Fast-forward
 .../augmentation/12_export_augmented_recbole.py    | 72 +++++++++++++++++-----
 1 file changed, 55 insertions(+), 17 deletions(-)


In [ ]:
from pathlib import Path

exports = Path("datasets/Video Games/exports")

for f in [
    "Video_Games_user_histories.json",
    "Video_Games_augmented_user_histories.json",
    "Video_Games_llama_augmented_user_histories.json",
]:
    path = exports / f
    print(f"{'OK' if path.exists() else 'MISSING'}: {path}")

OK: datasets/Video Games/exports/Video_Games_user_histories.json
OK: datasets/Video Games/exports/Video_Games_augmented_user_histories.json
OK: datasets/Video Games/exports/Video_Games_llama_augmented_user_histories.json


In [ ]:
!python pipeline/augmentation/14_filter_short_history_augmentation.py --original "datasets/Video Games/exports/Video_Games_user_histories.json" --augmented-input "datasets/Video Games/exports/Video_Games_augmented_user_histories.json" --output "datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories.json" --threshold 7

VIDEO GAMES — SELECTIVE LLM AUGMENTATION
Threshold: original history <= 7
Input augmented dataset: Video_Games_augmented_user_histories.json
Output: Video_Games_selective_qwen_augmented_user_histories.json

STEP 1 — READING ORIGINAL HISTORY LENGTHS
Streaming: datasets/Video Games/exports/Video_Games_user_histories.json
  Processed 100,000 users...

Original users processed: 109,258

STEP 2 — VALIDATING SOURCE AUGMENTED DATASET
Streaming: datasets/Video Games/exports/Video_Games_augmented_user_histories.json
  Processed 100,000 users...

Augmented users processed: 109,258

Synthetic interactions per user:
  3: 109,258 users

Source validation: PASSED

STEP 3 — CREATING SELECTIVE DATASET
Input:  datasets/Video Games/exports/Video_Games_augmented_user_histories.json
Output: datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories.json
Threshold: 7
  Processed 100,000 users | short=62,022 | long=37,978

FILTERING COMPLETE
Users processed: 109,258
Users with original

In [ ]:
%%writefile pipeline/augmentation/15_fix_sequence_positions.py

import argparse
import json
import os
import ijson


def fix_positions(input_path, output_path):

    if os.path.exists(output_path):
        raise FileExistsError(
            f"Output already exists: {output_path}"
        )

    users = 0
    interactions = 0

    print("=" * 70)
    print("FIXING SEQUENCE POSITIONS")
    print("=" * 70)
    print(f"Input:  {input_path}")
    print(f"Output: {output_path}")
    print()

    with open(input_path, "rb") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:

        outfile.write("{")
        first_user = True

        for user_id, sequence in ijson.kvitems(infile, ""):

            if not first_user:
                outfile.write(",")

            first_user = False

            fixed_sequence = []

            for position, item in enumerate(sequence, start=1):
                item = dict(item)
                item["sequence_position"] = position
                fixed_sequence.append(item)

            outfile.write(
                json.dumps(
                    str(user_id)
                )
            )
            outfile.write(":")
            outfile.write(
                json.dumps(
                    fixed_sequence,
                    ensure_ascii=False
                )
            )

            users += 1
            interactions += len(fixed_sequence)

            if users % 10000 == 0:
                print(
                    f"Processed {users:,} users..."
                )

        outfile.write("}")

    print()
    print("=" * 70)
    print("COMPLETE")
    print("=" * 70)
    print(f"Users:        {users:,}")
    print(f"Interactions: {interactions:,}")
    print(f"Output:       {output_path}")


def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input",
        required=True
    )

    parser.add_argument(
        "--output",
        required=True
    )

    args = parser.parse_args()

    fix_positions(
        args.input,
        args.output
    )


if __name__ == "__main__":
    main()

Overwriting pipeline/augmentation/15_fix_sequence_positions.py


In [ ]:
!python pipeline/augmentation/15_fix_sequence_positions.py \
  --input "datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories.json" \
  --output "datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories_fixed.json"

FIXING SEQUENCE POSITIONS
Input:  datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories.json
Output: datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories_fixed.json

Processed 10,000 users...
Processed 20,000 users...
Processed 30,000 users...
Processed 40,000 users...
Processed 50,000 users...
Processed 60,000 users...
Processed 70,000 users...
Processed 80,000 users...
Processed 90,000 users...
Processed 100,000 users...

COMPLETE
Users:        109,258
Interactions: 1,152,882
Output:       datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories_fixed.json


In [ ]:
!python pipeline/augmentation/15_fix_sequence_positions.py \
  --input "datasets/Video Games/exports/Video_Games_selective_llama_augmented_user_histories.json" \
  --output "datasets/Video Games/exports/Video_Games_selective_llama_augmented_user_histories_fixed.json"

FIXING SEQUENCE POSITIONS
Input:  datasets/Video Games/exports/Video_Games_selective_llama_augmented_user_histories.json
Output: datasets/Video Games/exports/Video_Games_selective_llama_augmented_user_histories_fixed.json

Processed 10,000 users...
Processed 20,000 users...
Processed 30,000 users...
Processed 40,000 users...
Processed 50,000 users...
Processed 60,000 users...
Processed 70,000 users...
Processed 80,000 users...
Processed 90,000 users...
Processed 100,000 users...

COMPLETE
Users:        109,251
Interactions: 1,152,822
Output:       datasets/Video Games/exports/Video_Games_selective_llama_augmented_user_histories_fixed.json


In [ ]:
!python pipeline/augmentation/12_export_augmented_recbole.py \
  --dataset video_games \
  --input-file "datasets/Video Games/exports/Video_Games_selective_qwen_augmented_user_histories_fixed.json" \
  --output-file "datasets/Video Games/exports/Video_Games_selective_qwen_augmented.inter" \
  --selective

EXPORTING AUGMENTED VIDEO GAMES DATA TO RECBOLE

Streaming user histories...

VALIDATING RECBOLE EXPORT
Selective augmentation: 69,074 users have 3 synthetic interactions.
Export validation passed.
Every user has exactly one validation interaction.
Every user has exactly one test interaction.
69,074 users have exactly 3 LLM synthetic interactions.
40,184 users have no synthetic interactions.

Writing RecBole interaction file:
datasets/Video Games/exports/Video_Games_selective_qwen_augmented.inter

AUGMENTED RECBOLE EXPORT COMPLETE
Dataset:                Video Games
Users:                  109,258
Real train interactions: 727,144
Synthetic interactions: 207,222
Validation interactions: 109,258
Test interactions:      109,258
Total interactions:     1,152,882

Output file:
datasets/Video Games/exports/Video_Games_selective_qwen_augmented.inter

FIRST 10 EXPORTED ROWS
('AEVPPTMG43C6GWSR7I2UGRQN7WFQ', 'B08R5B7YS4', 1.0)
('AEVPPTMG43C6GWSR7I2UGRQN7WFQ', 'B0863MT183', 2.0)
('AEVPPTMG43C6GWS

In [ ]:
!python pipeline/augmentation/12_export_augmented_recbole.py \
  --dataset video_games \
  --input-file "datasets/Video Games/exports/Video_Games_selective_llama_augmented_user_histories_fixed.json" \
  --output-file "datasets/Video Games/exports/Video_Games_selective_llama_augmented.inter" \
  --selective

EXPORTING AUGMENTED VIDEO GAMES DATA TO RECBOLE

Streaming user histories...

VALIDATING RECBOLE EXPORT
Selective augmentation: 69,069 users have 3 synthetic interactions.
Export validation passed.
Every user has exactly one validation interaction.
Every user has exactly one test interaction.
69,069 users have exactly 3 LLM synthetic interactions.
40,182 users have no synthetic interactions.

Writing RecBole interaction file:
datasets/Video Games/exports/Video_Games_selective_llama_augmented.inter

AUGMENTED RECBOLE EXPORT COMPLETE
Dataset:                Video Games
Users:                  109,251
Real train interactions: 727,113
Synthetic interactions: 207,207
Validation interactions: 109,251
Test interactions:      109,251
Total interactions:     1,152,822

Output file:
datasets/Video Games/exports/Video_Games_selective_llama_augmented.inter

FIRST 10 EXPORTED ROWS
('AEVPPTMG43C6GWSR7I2UGRQN7WFQ', 'B08R5B7YS4', 1.0)
('AEVPPTMG43C6GWSR7I2UGRQN7WFQ', 'B0863MT183', 2.0)
('AEVPPTMG43C6G